In [1]:
import pandas as pd
import numpy as np 

# csv파일을 불러옴
df1 = pd.read_csv('../data/tmdb_5000_credits.csv')
df2 = pd.read_csv('../data/tmdb_5000_movies.csv')
df1.columns = ['id', 'title', 'cast', 'crew']

# 두 csv파일을 하나로 합침(중복된 제목은 빼고)
df = df2.merge(df1[['id', 'cast', 'crew']], on='id')

In [2]:
df['overview'].head(1)

0    In the 22nd century, a paraplegic Marine is di...
Name: overview, dtype: object

In [3]:
# tfidf : 문서 전체에 많이 나오는 단어는 감점. 특정 문서에서 많이 나오는 단어는 가점 
# 전처리
# 내용이 비어있는 행이 있는지 확인
df['overview'] = df['overview'].fillna('')

In [4]:
# 모델 객체 생성 
from sklearn.feature_extraction.text import TfidfVectorizer
# stop_words : a, this와 같이 불필요한 단어를 제거
tfidf = TfidfVectorizer(stop_words='english')
# from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
# ENGLISH_STOP_WORDS
# tfidf 매트릭스 생성
tfidf_matrix = tfidf.fit_transform(df['overview'])

In [ ]:
# 코사인 유사도를 계산
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [6]:
# df의 idx를 값으로 보내고 df의 title를 새로운 시리즈의 idx로 보냄
# 왜? 주어진 제목의 idx를 편하게 찾으려고 
indices = pd.Series(df.index, index=df2['title']).drop_duplicates()
# indices['Avatar']
# df.loc[df['title'] == 'Avatar'].index

In [ ]:
# 영화 제목이 주어지면 주어진 영화 제목과 내용이 유사한 영화 10개를 추천하는 함수
def get_recommendations(title, cosine_sim=cosine_sim):
	# 영화 제목으로 idx를 찾음(왜? 해당 영화의 코사인 유사도를 가져오기 위해)
	# 코사인 유사도는 idx를 이용해서 찾아야 함
	idx = indices[title]
	# 코사인 유사도를 정렬하려고 하는데 그냥 정렬하면 기존 위치가 섞임 
	# => 유사도만 보면 어떤 영화인지 모름
	# => 코사인 유도를 (idx, 코사인유사도값)으로 된 튜플을 만들면 섞여도 idx를 알 수 있음
	sim_scores = list(enumerate(cosine_sim[idx]))
	# 코사인 유사도를 기준으로 정렬(람다함수 활용)
	sim_scores = sorted(sim_scores, key=lambda x : x[1], reverse=True)
	# 0번지는 검색 영화이기 때문에 0번지를 제외한 10개 추출
	sim_scores = sim_scores[1:11]
	# 여기서부터는 코사인 유사도 값이 필요없기 때문에 영화 idx만 추출해서 리스트로 만듬
	movie_indices = [i[0] for i in sim_scores]
	# 영화 idx를 이용해서 영화 제목들을 가져와서 반환
	return df['title'].iloc[movie_indices]

In [8]:
idx = indices['Avatar']
sim_scores = list(enumerate(cosine_sim[idx])) # 영화 Avatar의 코사인 유사도
sim_scores = sorted(sim_scores, key=lambda x : x[1], reverse=True)

In [9]:
# 람다함수를 설명하는 예제
lst = ['idx', 'value']
# 간단한 함수
def test(x):
	return x[1]
test(lst)

'value'

In [10]:
# 간단한 함수 test를 람다함수로 만들고 바로 테스트
(lambda x: x[1])(lst)

'value'

In [11]:
sim_scores = sim_scores[1:11]

In [12]:
movie_indices = [i[0] for i in sim_scores]


In [13]:
df['title'].iloc[movie_indices]

3604                       Apollo 18
2130                    The American
634                       The Matrix
1341            The Inhabited Island
529                 Tears of the Sun
1610                           Hanna
311     The Adventures of Pluto Nash
847                         Semi-Pro
775                        Supernova
2628             Blood and Chocolate
Name: title, dtype: object

In [14]:
get_recommendations('Batman Forever')

3                         The Dark Knight Rises
119                               Batman Begins
65                              The Dark Knight
428                              Batman Returns
210                              Batman & Robin
3854    Batman: The Dark Knight Returns, Part 2
1359                                     Batman
4343                                   Cry_Wolf
174                         The Incredible Hulk
9            Batman v Superman: Dawn of Justice
Name: title, dtype: object